In [1]:
import data.breathe_data as bd
import data.helpers as dh
from plotly.subplots import make_subplots
import pandas as pd
import plotly.graph_objs as go
import cfr.corr as corr

DARK_GREEN = "#006400"
BLUE = "#1f77b4"

In [ ]:
# df1 = bd.load_meas_from_excel(
#     # "infer_all_19_data_with_best_FEV1",
#     "pppfev1_ppfev1st_ppfev1ft_IV_19_plus1820_assoc",
#     study_folder="CFR",
#     str_cols_to_arrays=[
#         # "Airway resistance (%)",
#         # "P(HFEV1|FEF2575, bFEV1, FEV1)",
#         "P(HFEV1|bFEV1)",
#         "P(HFEV1|FEV1)",
#     ],
#     use_csv=True,
#     bypass_sanity_checks=True,
# )

In [6]:
df = bd.load_meas_from_excel(
    # "infer_all_19_data_with_best_FEV1",
    "ppfev1st_ft_bFEV1_2016-19_IV_2019-21_assoc",
    # "ppfev1st_ft_bFEV1_2012-15_IV_2015-18_assoc",
    study_folder="CFR",
    str_cols_to_arrays=[
        # "Airway resistance (%)",
        # "P(HFEV1|FEF2575, bFEV1, FEV1)",
        "P(HFEV1|bFEV1)",
        "P(HFEV1|FEV1)",
    ],
    use_csv=True,
    bypass_sanity_checks=True,
)

In [8]:
df = bd.calc_predicted_FEV1_LMS_df(df)
df = bd.calc_FEV1_prct_predicted_df(df, with_ecFEV1=False)

# Methods viz: truncating HFEV1

In [6]:
import src.inference.helpers as ih
import src.models.helpers as mh
import numpy as np

In [ ]:
fig = make_subplots(2, 1, shared_xaxes=False)

hfev1_prior = {"type": "default", "height": 173, "age": 35, "sex": "Female"}
HFEV1 = mh.VariableNode("Healthy FEV1 (L)", 1, 6, 0.05, prior=hfev1_prior)

xmin = 1.9
xmax = 5
ih.plot_histogram(fig, HFEV1, HFEV1.cpt, xmin, xmax, 1, 1, colour=BLUE)

hist_max = 4
hfev1_trunc = HFEV1.cpt.copy()
bin_idx = np.argmin(HFEV1.bins < hist_max + 0.001) - 1
hfev1_trunc[0:bin_idx] = 0
hfev1_trunc /= hfev1_trunc.sum()
ih.plot_histogram(fig, HFEV1, hfev1_trunc, xmin, xmax, 2, 1, colour=DARK_GREEN)

mean_full = HFEV1.get_mean(HFEV1.cpt)
mean_trunc = HFEV1.get_mean(hfev1_trunc)

fig.update_xaxes(
    tickvals=[round(mean_full, 2)],
    ticktext=[f"{mean_full:.2f}"],
    row=1,
    col=1,
    range=[1.9, 5],
)
fig.update_xaxes(
    title="FEV1 (L)",
    tickvals=[2, 3, round(mean_trunc, 2), 4, 5],
    ticktext=[2, 3, f"{mean_trunc:.2f}", 4, 5],
    row=2,
    col=1,
)

fig.update_yaxes(showticklabels=False, ticks="", row=1, col=1)
fig.update_yaxes(showticklabels=False, ticks="", row=2, col=1)

title = "Healthy FEV1 distribution (full vs truncated)"
fig.update_layout(
    width=500,
    height=400,
    template="simple_white",
    showlegend=False,
    title=title,
)

fig.show()
# fig.write_image(dh.get_path_to_main() + f"PlotsCFR/{title}.pdf")

# Viz ranked associations with IV day

In [10]:
df.columns

Index(['ID', 'Age', 'Height', 'FEV1', 'FEF2575', 'best FEV1', 'Sex',
       'Date Recorded', 'ecFEV1', 'ecFEF2575', 'ecFEF2575%ecFEV1',
       'Predicted FEV1', 'ecFEV1 % Predicted', 'FEV1 % Predicted',
       'best FEV1 old', 'idx FEV1', 'idx FEF2575%FEV1', 'idx best FEV1',
       'best FEV1 2016-19', 'best FEV1 year', 'bFEV1 diff', 'bFEV1 % diff',
       'idx best FEV1 2016-19', 'P(HFEV1|FEV1)', 'P(HFEV1|bFEV1)',
       'FEV1%PredST', 'FEV1%PredFT', 'ppFEV1FT - ppFEV1ST', 'IVs', 'IV days'],
      dtype='object')

In [11]:
import numpy as np

# --- Configuration ---
diff_col = "ppFEV1FT - ppFEV1ST"
prctile = 50
baseline_col = "FEV1 % Predicted"
predicted_col = "FEV1%PredFT"
iv_col = "IV days"
# iv_col = "IV days 2015-17"
# iv_col = "IV days 2015-18"
# iv_col = "IV days 2015-16"
# iv_col = "IV days 2015"
N_BOOTSTRAP = 10000
title = f"Ranked IV days by severity, |{diff_col}| > {prctile}th pctile, bFEV1 2016-19, IV 2019-21 bis"
# title = f"Ranked IV days by severity 2, |{diff_col}| > {prctile}th pctile, bFEV1 2016-19, IV 2019-21 log"
# title = f"Ranked IV days by severity, |{diff_col}| > {prctile}th pctile, bFEV1 2012-15, IV 2015-18 bis"
# title = f"Ranked IV days by severity, |{diff_col}| > {prctile}th pctile, bFEV1 2012-15, IV 2015-18"

# --- Data preparation ---|
df_filt = df.copy()
threshold = df_filt[diff_col].abs().quantile(prctile / 100)
df_filt = df_filt[df_filt[diff_col].abs() > threshold]

df_filt.loc[df_filt[baseline_col] >= 70, "severity"] = "mild"
df_filt.loc[
    (df_filt[baseline_col] >= 40) & (df_filt[baseline_col] < 70), "severity"
] = "moderate"

df_mild = df_filt[df_filt["severity"] == "mild"]
df_moderate = df_filt[df_filt["severity"] == "moderate"]
iv_max = df_filt[iv_col].max() * 1.05

# --- Correlations ---
res_mild = corr.pred_vs_baseline_corr_diff_test(
    df_mild[predicted_col],
    df_mild[baseline_col],
    df_mild[iv_col],
    n_bootstrap=N_BOOTSTRAP,
)
res_moderate = corr.pred_vs_baseline_corr_diff_test(
    df_moderate[predicted_col],
    df_moderate[baseline_col],
    df_moderate[iv_col],
    n_bootstrap=N_BOOTSTRAP,
)


# --- Scatter plot function ---
def plot_scatter_panel(fig, df_sev, rank_metric, row, col):
    """
    Plot IV days vs rank index for df_sev sorted by rank_metric (descending).
    Adds a dotted grey vertical line on col 1 (mild) where rank_metric crosses 70%.
    """
    df_sorted = df_sev.sort_values(rank_metric, ascending=False).reset_index(drop=True)
    x_rank = list(range(len(df_sorted)))
    color = DARK_GREEN if rank_metric == predicted_col else BLUE

    fig.add_trace(
        go.Scatter(
            x=x_rank,
            y=df_sorted[iv_col],
            mode="markers",
            marker=dict(size=3, color=color),
            customdata=df_sorted["ID"],
            hovertemplate="ID: %{customdata}<br>IV days: %{y:.0f}<extra></extra>",
            showlegend=False,
        ),
        row=row,
        col=col,
    )

    # Dotted grey vertical line at the 70% boundary (mild plots only)
    # if col == 1:
    #     above = df_sorted[rank_metric] >= 70
    #     if above.any() and not above.all():
    #         last_above_idx = int(np.where(above.values)[0][-1])
    #         fig.add_vline(
    #             x=last_above_idx + 0.5,
    #             line_dash="dot",
    #             line_color="grey",
    #             line_width=1.5,
    #             row=row,
    #             col=col,
    #         )

    title_suff = (
        "personalised FEV1%pred."
        if rank_metric == "FEV1%PredFT"
        else "baseline FEV1%pred."
    )
    fig.update_xaxes(
        showticklabels=False,
        title_text=f"Individuals ranked by {title_suff}",
        linecolor="black",
        linewidth=1,
        showline=True,
        row=row,
        col=col,
    )
    fig.update_yaxes(
        title_text=iv_col,
        type="log",
        range=[-1, np.log10(iv_max)],
        linecolor="black",
        linewidth=1,
        showline=True,
        row=row,
        col=col,
    )


# --- Figure ---
fig = make_subplots(
    rows=2,
    cols=4,
    vertical_spacing=0.12,
    horizontal_spacing=0,
    column_widths=[2, 1, 2, 1],
    column_titles=["Mild CF", "", "Moderate CF", ""],
    row_titles=["Personalised", "Baseline"],
)

# Row 1: ranked by personalised predicted FEV1 (FEV1%PredFT)
plot_scatter_panel(fig, df_mild, predicted_col, row=1, col=1)
plot_scatter_panel(fig, df_moderate, predicted_col, row=1, col=3)

# Row 2: ranked by standard predicted FEV1 (FEV1 % Predicted)
plot_scatter_panel(fig, df_mild, baseline_col, row=2, col=1)
plot_scatter_panel(fig, df_moderate, baseline_col, row=2, col=3)


# --- Correlation annotations ---
subplot_centres = {
    (1, 1): (0.39, 0.78),
    (2, 1): (0.39, 0.22),
    (1, 3): (0.9, 0.78),
    (2, 3): (0.9, 0.22),
}

for (r, c), res in [
    ((1, 1), res_mild),
    ((1, 3), res_moderate),
    ((2, 1), res_mild),
    ((2, 3), res_moderate),
]:
    x_paper, y_paper = subplot_centres[(r, c)]
    if r == 1:
        value = res["r_baseline"]
        pval = res["p_baseline"]
    else:
        value = res["r_predicted"]
        pval = res["p_predicted"]

    fig.add_annotation(
        xref="paper",
        yref="paper",
        x=x_paper,
        y=y_paper,
        text=f"corr={value:+.3f}<br>p={pval:.2e}",
        showarrow=False,
        align="center",
        xanchor="center",
        yanchor="middle",
        font=dict(size=10),
    )

ci_data_boot = f"<br>p-value boot - Mild {res_mild['p_val_bootstrapped_corr_diff']:.4f}, Moderate {res_moderate['p_val_bootstrapped_corr_diff']:.4f}"
ci_data_perm = f"| p-value perm - Mild {res_mild['p_val_perm_test_corr_diff']:.4f}, Moderate {res_moderate['p_val_perm_test_corr_diff']:.4f}"
fig.update_layout(
    height=700,
    width=1200,
    title=title + ci_data_boot + ci_data_perm,
    template="simple_white",
    plot_bgcolor="white",
    paper_bgcolor="white",
)

# fig.show()
fig.write_image(dh.get_path_to_main() + f"PlotsCFR/Ranked IV days and FEV1/{title}.pdf")

In [17]:
# QQ PLot - Check whether correlations difference are gaussian distributioed

arr = np.asarray(res_mild["diffs"])
arr_sorted = np.sort(arr)
n = arr_sorted.size
if n == 0:
    raise ValueError("Empty array for QQ plot")

# Use sample mean/std; handle zero-std
mu = arr_sorted.mean()
sigma = arr_sorted.std(ddof=0)
probs = (np.arange(1, n + 1) - 0.5) / n

# Approximate theoretical quantiles by sampling from the matching normal
rng = np.random.default_rng(0)
theor_sample = rng.normal(loc=mu, scale=sigma if sigma > 0 else 1.0, size=200_000)
theoretical = np.quantile(theor_sample, probs)

fig_qq = go.Figure()
fig_qq.add_trace(
    go.Scatter(
        x=theoretical,
        y=arr_sorted,
        mode="markers",
        marker=dict(size=5, color="darkblue"),
        hovertemplate="theor: %{x:.4f}<br>sample: %{y:.4f}<extra></extra>",
        name="Quantiles",
    )
)

# diagonal y=x line
minv = min(theoretical.min(), arr_sorted.min())
maxv = max(theoretical.max(), arr_sorted.max())
fig_qq.add_trace(
    go.Scatter(
        x=[minv, maxv],
        y=[minv, maxv],
        mode="lines",
        line=dict(color="red", dash="dash"),
        showlegend=False,
        hoverinfo="skip",
    )
)

fig_qq.update_layout(
    title="QQ plot of res_mild['diffs'] vs Normal",
    xaxis_title="Theoretical quantiles",
    yaxis_title="Sample quantiles",
    template="simple_white",
    width=700,
    height=500,
)
fig_qq.show()

# Viz ranked associations with IV days

In [ ]:
diff_col = "ppFEV1FT - ppFEV1ST"
prctile = 50  # keep rows where abs(diff_col) > this percentile threshold

df_ranked = df.copy()
df_ranked = df_ranked[(df_ranked["ecFEF2575%ecFEV1"] < 70)].copy()
threshold = df_ranked[diff_col].abs().quantile(prctile / 100)
df_ranked = df_ranked[df_ranked[diff_col].abs() > threshold]

rank_metric = "FEV1 % Predicted"

df_ranked.loc[df_ranked[rank_metric] >= 70, "severity"] = "mild"
df_ranked.loc[
    (df_ranked[rank_metric] >= 40) & (df_ranked[rank_metric] < 70), "severity"
] = "moderate"
df_ranked.loc[df_ranked[rank_metric] < 40, "severity"] = "severe"

iv_max = df_ranked["IV days"].max() * 1.05

# severities = [("mild", 1, 2), ("moderate", 3, 4), ("severe", 5, 6)]
severities = [("mild", 1, 2)]
fev_metrics = ["FEV1%PredST", "FEV1%PredFT"]
fev_colors = {"FEV1%PredST": "blue", "FEV1%PredFT": "red"}
iv_color = "rgba(64, 64, 64, 0.8)"

# Compute shared y range per severity across both fev_metrics
fev_range_by_severity = {}
for severity_label, _, _ in severities:
    df_sev = df_ranked[df_ranked["severity"] == severity_label]
    vals = pd.concat([df_sev[m] for m in fev_metrics]).dropna()
    pad = (vals.max() - vals.min()) * 0.05
    fev_range_by_severity[severity_label] = [vals.min() - pad, vals.max() + pad]


def add_ranked_fev_iv_panels(
    fig,
    df_severity,
    fev_metric,
    severity_label,
    row_fev,
    row_iv,
    col,
    overlay_metric=None,
):
    df_sorted = df_severity.sort_values(fev_metric, ascending=False).reset_index(
        drop=True
    )
    x_rank = list(range(len(df_sorted)))

    if overlay_metric is not None:
        fig.add_trace(
            go.Scatter(
                x=x_rank,
                y=df_sorted[overlay_metric],
                mode="markers",
                marker=dict(size=3, opacity=0.3, color=fev_colors[overlay_metric]),
                customdata=df_sorted["ID"],
                hovertemplate="ID: %{customdata}<br>"
                + overlay_metric
                + ": %{y:.1f}<extra></extra>",
                showlegend=False,
            ),
            row=row_fev,
            col=col,
        )

    fig.add_trace(
        go.Scatter(
            x=x_rank,
            y=df_sorted[fev_metric],
            mode="markers",
            marker=dict(size=3, opacity=1.0, color=fev_colors[fev_metric]),
            customdata=df_sorted["ID"],
            hovertemplate="ID: %{customdata}<br>"
            + fev_metric
            + ": %{y:.1f}<extra></extra>",
            showlegend=False,
        ),
        row=row_fev,
        col=col,
    )
    fig.add_trace(
        go.Scatter(
            x=x_rank,
            y=df_sorted["IV days"],
            mode="markers",
            marker=dict(size=3, color=iv_color),
            customdata=df_sorted["ID"],
            hovertemplate="ID: %{customdata}<br>IV days: %{y:.0f}<extra></extra>",
            showlegend=False,
        ),
        row=row_iv,
        col=col,
    )
    iv_max = df_sorted["IV days"].max() * 1.05
    fig.update_xaxes(
        showticklabels=False,
        linecolor="black",
        linewidth=1,
        showline=True,
        row=row_fev,
        col=col,
    )
    fig.update_xaxes(
        showticklabels=False,
        linecolor="black",
        linewidth=1,
        showline=True,
        row=row_iv,
        col=col,
    )
    fig.update_yaxes(
        title_text=f"{fev_metric}",
        linecolor="black",
        linewidth=1,
        showline=True,
        range=fev_range_by_severity[severity_label],
        row=row_fev,
        col=col,
    )
    fig.update_yaxes(
        title_text="IV days",
        range=[-5, iv_max],
        linecolor="black",
        linewidth=1,
        showline=True,
        row=row_iv,
        col=col,
    )


fig = make_subplots(
    rows=2,
    cols=2,
    vertical_spacing=0.03,
    horizontal_spacing=0.12,
    column_titles=["Baseline", "Predicted"],
    row_titles=[
        "Mild CF",
        "",
        "Moderate CF",
        "",
        "Severe CF",
        "",
    ],
)

for col_idx, fev_metric in enumerate(fev_metrics, start=1):
    overlay = "FEV1%PredST" if col_idx == 2 else None
    for severity_label, row_iv, row_fev in severities:
        df_sev = df_ranked[df_ranked["severity"] == severity_label]
        add_ranked_fev_iv_panels(
            fig, df_sev, fev_metric, severity_label, row_fev, row_iv, col_idx
        )
        add_ranked_fev_iv_panels(
            fig,
            df_sev,
            fev_metric,
            severity_label,
            row_fev,
            row_iv,
            col_idx,
            overlay_metric=overlay,
        )

title = f"Ranked FEV1 and IV days by severity (|{diff_col}| > {prctile}th pctile) 2"
fig.update_layout(
    height=700,
    width=1100,
    # height=1100, width=1100,
    title=title,
    template="simple_white",
    plot_bgcolor="white",
    paper_bgcolor="white",
)
fig.write_image(dh.get_path_to_main() + f"PlotsCFR/Ranked FEV1 and IV days/{title}.pdf")

## Archive

In [ ]:
# Archive
diff_col = "ppFEV1FT - ppFEV1ST"
prctile = 0  # keep rows where abs(diff_col) > this percentile threshold

df_ranked = df.copy()
# df_ranked = df_ranked[(df_ranked["ecFEF2575%ecFEV1"] < 70)].copy()
threshold = df_ranked[diff_col].abs().quantile(prctile / 100)
df_ranked = df_ranked[df_ranked[diff_col].abs() > threshold]

df_ranked.loc[df_ranked["FEV1%PredST"] >= 70, "severity"] = "mild"
df_ranked.loc[
    (df_ranked["FEV1%PredST"] >= 40) & (df_ranked["FEV1%PredST"] < 70), "severity"
] = "moderate"
df_ranked.loc[df_ranked["FEV1%PredST"] < 40, "severity"] = "severe"

iv_max = df_ranked["IV days"].max() * 1.05

severities = [("mild", 1, 2), ("moderate", 3, 4), ("severe", 5, 6)]
fev_metrics = ["FEV1%PredST", "FEV1%PredFT"]
fev_colors = {"FEV1%PredST": "blue", "FEV1%PredFT": "red"}
iv_color = "rgba(64, 64, 64, 0.8)"

# Compute shared y range per severity across both fev_metrics
fev_range_by_severity = {}
for severity_label, _, _ in severities:
    df_sev = df_ranked[df_ranked["severity"] == severity_label]
    vals = pd.concat([df_sev[m] for m in fev_metrics]).dropna()
    pad = (vals.max() - vals.min()) * 0.05
    fev_range_by_severity[severity_label] = [vals.min() - pad, vals.max() + pad]


def add_ranked_fev_iv_panels(
    fig,
    df_severity,
    fev_metric,
    severity_label,
    row_fev,
    row_iv,
    col,
    overlay_metric=None,
):
    df_sorted = df_severity.sort_values(fev_metric, ascending=False).reset_index(
        drop=True
    )
    x_rank = list(range(len(df_sorted)))

    if overlay_metric is not None:
        fig.add_trace(
            go.Scatter(
                x=x_rank,
                y=df_sorted[overlay_metric],
                mode="markers",
                marker=dict(size=3, opacity=0.3, color=fev_colors[overlay_metric]),
                customdata=df_sorted["ID"],
                hovertemplate="ID: %{customdata}<br>"
                + overlay_metric
                + ": %{y:.1f}<extra></extra>",
                showlegend=False,
            ),
            row=row_fev,
            col=col,
        )

    fig.add_trace(
        go.Scatter(
            x=x_rank,
            y=df_sorted[fev_metric],
            mode="markers",
            marker=dict(size=3, opacity=1.0, color=fev_colors[fev_metric]),
            customdata=df_sorted["ID"],
            hovertemplate="ID: %{customdata}<br>"
            + fev_metric
            + ": %{y:.1f}<extra></extra>",
            showlegend=False,
        ),
        row=row_fev,
        col=col,
    )
    fig.add_trace(
        go.Scatter(
            x=x_rank,
            y=df_sorted["IV days"],
            mode="markers",
            marker=dict(size=3, color=iv_color),
            customdata=df_sorted["ID"],
            hovertemplate="ID: %{customdata}<br>IV days: %{y:.0f}<extra></extra>",
            showlegend=False,
        ),
        row=row_iv,
        col=col,
    )
    fig.update_xaxes(
        showticklabels=False,
        linecolor="black",
        linewidth=1,
        showline=True,
        row=row_fev,
        col=col,
    )
    fig.update_xaxes(
        showticklabels=False,
        linecolor="black",
        linewidth=1,
        showline=True,
        row=row_iv,
        col=col,
    )
    fig.update_yaxes(
        title_text=f"{fev_metric}",
        linecolor="black",
        linewidth=1,
        showline=True,
        range=fev_range_by_severity[severity_label],
        row=row_fev,
        col=col,
    )
    fig.update_yaxes(
        title_text="IV days",
        range=[-5, iv_max],
        linecolor="black",
        linewidth=1,
        showline=True,
        row=row_iv,
        col=col,
    )


fig = make_subplots(
    rows=6,
    cols=2,
    vertical_spacing=0.03,
    horizontal_spacing=0.12,
    column_titles=["Baseline", "Predicted"],
    row_titles=[
        "Mild CF",
        "",
        "Moderate CF",
        "",
        "Severe CF",
        "",
    ],
)

for col_idx, fev_metric in enumerate(fev_metrics, start=1):
    overlay = "FEV1%PredST" if col_idx == 2 else None
    for severity_label, row_fev, row_iv in severities:
        df_sev = df_ranked[df_ranked["severity"] == severity_label]
        add_ranked_fev_iv_panels(
            fig, df_sev, fev_metric, severity_label, row_fev, row_iv, col_idx
        )
        add_ranked_fev_iv_panels(
            fig,
            df_sev,
            fev_metric,
            severity_label,
            row_fev,
            row_iv,
            col_idx,
            overlay_metric=overlay,
        )

title = f"Ranked FEV1 and IV days by severity (|{diff_col}| > {prctile}th pctile) 2"
fig.update_layout(
    height=1100,
    width=900,
    title=title,
    template="simple_white",
    plot_bgcolor="white",
    paper_bgcolor="white",
)
# fig.write_image(dh.get_path_to_main() + f"PlotsCFR/{title}.pdf")

## Evaluate reclassified individuals

In [19]:
df_mild = df_ranked[df_ranked["severity"] == "mild"]
df_reclassified = df_mild[df_ranked["FEV1%PredFT"] < 70]

/var/folders/zq/v2r6yn111s3gpdf8lzf72xvw0000gn/T/ipykernel_13480/2390661838.py:2: UserWarning:

Boolean Series key will be reindexed to match DataFrame index.



In [ ]:
print(f"{df_reclassified.shape[0]} of mild patients reclassified as moderate")
print(
    f"{df_reclassified.shape[0]/df_mild.shape[0] * 100:.1f}% of mild patients reclassified as moderate"
)
df_reclassified["FEV1%PredFT"].describe()

74 of mild patients reclassified as moderate
9.1% of mild patients reclassified as moderate


count    74.000000
mean     67.148786
std       3.250502
min      55.578116
25%      66.642265
50%      68.216090
75%      69.345391
max      69.978072
Name: FEV1%PredFT, dtype: float64

In [22]:
df_reclassified.sort_values(by="FEV1%PredFT").head(5)

,ID,Age,Height,FEV1,FEF2575,best FEV1,Sex,Date Recorded,ecFEV1,ecFEF2575,...,bFEV1 % diff,idx best FEV1 2016-19,P(HFEV1|FEV1),P(HFEV1|bFEV1),FEV1%PredST,FEV1%PredFT,ppFEV1FT - ppFEV1ST,IVs,IV days,severity
66,B157445,18,173,3.22,2.26,3.61,Male,2019-01-01,3.22,2.26,...,58.448758,114,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.266...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",75.885290,55.578116,-20.307174,0.000000,0.000000,mild
34,B157861,50,170,2.84,2.30,2.84,Female,2019-01-01,2.84,2.30,...,76.760569,100,"[0.0, 0.0, 1.23407231e-306, 1.07774138e-288, 2...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",88.558334,56.391122,-32.167211,0.000000,0.000000,mild
52,B164053,34,166,2.79,1.55,2.92,Female,2019-01-01,2.79,1.55,...,61.986297,94,"[0.0, 8.60667128e-312, 1.30789243e-293, 5.5568...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",83.881837,58.466162,-25.415676,0.333333,5.000000,mild
192,B167687,26,167,2.74,1.43,3.39,Female,2019-01-01,2.74,1.43,...,32.743359,90,"[6.19355526e-316, 1.51329715e-297, 1.02024786e...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",79.490345,59.359564,-20.130781,5.666667,90.000000,mild
188,B167849,29,158,2.25,1.23,2.73,Female,2019-01-01,2.25,1.23,...,33.333332,72,"[5.16854868e-190, 3.8968373e-175, 7.00232658e-...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",74.789247,60.035783,-14.753465,2.000000,21.333333,mild


# Correlations computation

In [34]:
import cfr.corr as corr

In [25]:
df.columns

Index(['ID', 'Sex', 'Height', 'FEV1', 'best FEV1', 'Date Recorded', 'Age',
       'best FEV1 2012-15', 'best FEV1 year', 'IV days 2015',
       'IV days 2015-16', 'IV days 2015-17', 'IV days 2015-18', 'idx FEV1',
       'idx best FEV1 2012-15', 'P(HFEV1|FEV1)', 'P(HFEV1|bFEV1)',
       'FEV1%PredST', 'ppFEV1FT - ppFEV1ST', 'FEV1%PredFT', 'Predicted FEV1',
       'FEV1 % Predicted'],
      dtype='object')

In [ ]:
# --- Configuration ---
# baseline_col = "FEV1%PredST"
baseline_col = "FEV1 % Predicted"
predicted_col = "FEV1%PredFT"  # always FEV1%PredFT
iv_col = "IV days 2015-18"
# iv_col = "IV days"
diff_col = "ppFEV1FT - ppFEV1ST"
percentile_thresholds = [0, 25, 50, 75, 90]
N_BOOTSTRAP = 10_000

# Severity group definitions (based on baseline FEV1%PredST)
severity_groups = [
    ("Severe (<40)", df[baseline_col] < 40),
    ("Moderate (40-69)", (df[baseline_col] >= 40) & (df[baseline_col] < 70)),
    # ("Moderate 2 (40-49)", (df[baseline_col] >= 40) & (df[baseline_col] < 50)),
    # ("Moderate 1 (50-69)", (df[baseline_col] >= 50) & (df[baseline_col] < 70)),
    # ("Mild 3 (70-79)",     (df[baseline_col] >= 70) & (df[baseline_col] < 80)),
    # ("Mild 2 (80-89)",     (df[baseline_col] >= 80) & (df[baseline_col] < 90)),
    # ("Mild 1 (>=90)",       df[baseline_col] >= 90),
    ("Mild 1 (>=90)", df[baseline_col] >= 70),
]


def _analyze_subset(df_sub, label=""):
    n = len(df_sub)
    if n < 5:
        print(f"  [{label}] n={n}: skipping (too few samples)")
        return None
    res = corr.pred_vs_baseline_corr_diff_test(
        df_sub[baseline_col],
        df_sub[predicted_col],
        df_sub[iv_col],
        n_bootstrap=N_BOOTSTRAP,
    )
    print(
        f"  [{label}] n={res['n']:3d} | "
        f"r_base={res['r_baseline']:+.3f}(p={res['p_baseline']:.2e}), "
        f"r_pred={res['r_predicted']:+.3f}(p={res['p_predicted']:.2e}) | "
        f"diff={res['diff']:+.3f} | "
        f"p-value={res['p_val_corr_diffs']:.3f}"
    )
    return res


# --- Main Analysis ---
# all_results[prctile][label] = result dict
all_results = {}

for prctile in percentile_thresholds:
    print(f"\n{'='*100}")
    print(f"PERCENTILE THRESHOLD OF {diff_col}: {prctile}%")
    print(f"{'='*100}")
    all_results[prctile] = {}

    # Population level: threshold applied globally across all patients
    t_pop = df[diff_col].abs().quantile(prctile / 100)
    df_pop = df[df[diff_col].abs() >= t_pop].copy()
    print(f"\n  Population: abs({diff_col}) >= {t_pop:.4f}, n={len(df_pop)}")
    all_results[prctile]["Population"] = _analyze_subset(df_pop, label="Population")

    # By severity group: threshold applied within each group independently
    for group_name, group_mask in severity_groups:
        df_grp = df[group_mask].copy()
        if df_grp.empty:
            continue
        t_grp = df_grp[diff_col].abs().quantile(prctile / 100)
        df_sub = df_grp[df_grp[diff_col].abs() >= t_grp].copy()
        all_results[prctile][group_name] = _analyze_subset(df_sub, label=group_name)


PERCENTILE THRESHOLD OF ppFEV1FT - ppFEV1ST: 0%

  Population: abs(ppFEV1FT - ppFEV1ST) >= 0.00, n=4024
  [Population] n=4024 | r_base=-0.603(p=0.00e+00), r_pred=-0.604(p=0.00e+00) | diff=+0.001 | p-value=0.797
  [Severe (<40)] n=789 | r_base=-0.233(p=1.75e-11), r_pred=-0.236(p=9.30e-12) | diff=+0.003 | p-value=0.894
  [Moderate (40-69)] n=1590 | r_base=-0.285(p=1.76e-31), r_pred=-0.290(p=1.85e-32) | diff=+0.004 | p-value=0.852
  [Mild 1 (>=90)] n=1645 | r_base=-0.293(p=2.75e-34), r_pred=-0.308(p=7.34e-38) | diff=+0.015 | p-value=0.937

PERCENTILE THRESHOLD OF ppFEV1FT - ppFEV1ST: 25%

  Population: abs(ppFEV1FT - ppFEV1ST) >= 0.00, n=3018
  [Population] n=3018 | r_base=-0.533(p=1.07e-221), r_pred=-0.535(p=4.57e-223) | diff=+0.001 | p-value=0.700
  [Severe (<40)] n=592 | r_base=-0.298(p=6.76e-14), r_pred=-0.301(p=3.86e-14) | diff=+0.003 | p-value=0.793
  [Moderate (40-69)] n=1192 | r_base=-0.285(p=5.48e-24), r_pred=-0.286(p=3.83e-24) | diff=+0.001 | p-value=0.561
  [Mild 1 (>=90)] n=1

# Viz baseline-prediction diff vs IV days

In [ ]:
diff_col = "ppFEV1FT - ppFEV1ST"

dftmp = df.copy()

# Filter percentile of certain population
prctile = 90
for prctile in [50, 60, 70, 80, 90]:
    t = dftmp[diff_col].abs().quantile(prctile / 100)
    print(f"{prctile}th percentile of absolute difference: {t}")

    dftmp = dftmp[dftmp[diff_col].abs() > t]

    # Filter by severity level
    dftmp.loc[dftmp["FEV1%PredST"] >= 70, "severity"] = "mild"
    dftmp.loc[
        (dftmp["FEV1%PredST"] >= 40) & (dftmp["FEV1%PredST"] < 70), "severity"
    ] = "moderate"
    dftmp.loc[dftmp["FEV1%PredST"] < 40, "severity"] = "severe"

    fig = make_subplots(rows=3, cols=1, shared_xaxes=True)

    for i, severity in enumerate(["mild", "moderate", "severe"], start=1):
        dftmp_severity = dftmp[dftmp["severity"] == severity]
        fig.add_trace(
            go.Scatter(
                x=dftmp_severity[diff_col],
                y=dftmp_severity["IV days"],
                mode="markers",
                marker=dict(size=3, opacity=0.8),
            ),
            row=i,
            col=1,
        )
        fig.update_yaxes(
            title="IV days", range=[-5, dftmp["IV days"].max() * 1.1], row=i, col=1
        )
    fig.update_xaxes(title=f"{diff_col}", row=3, col=1)

    title = f"bFEV1_2016_19_IVdays_2019-21_{prctile}th_prctile"
    fig.update_layout(
        height=600,
        width=800,
        title=title,
    )

    fig.write_image(dh.get_path_to_main() + f"PlotsCFR/Viz diff vs IV days/{title}.pdf")

    ##########################################################################################
    def labels_from_bins(bins):
        labels = [f"< {bins[1]}"]
        for i in range(1, len(bins) - 2):
            labels.append(f"[{bins[i]}, {bins[i+1]})")
        labels.append(f">= {bins[-2]}")
        return labels

    fig = make_subplots(3, 1)

    row = 0
    for severity in ["mild", "moderate", "severe"]:
        row += 1
        dftmp2 = dftmp[dftmp["severity"] == severity].copy()
        bins = [-1000, -20, -11, -9, -7, -5, -3, -1, 0]
        labels = labels_from_bins(bins)
        dftmp2["diff_bin"] = pd.cut(dftmp2[diff_col], bins=bins, labels=labels)

        grouped = (
            dftmp2.groupby("diff_bin", observed=False)
            .agg(
                iv_mean=("IV days", "mean"),
                iv_std=("IV days", "std"),
                diff_mean=(diff_col, "mean"),
                count=(diff_col, "size"),
            )
            .reset_index()
        )

        fig.add_trace(
            go.Bar(
                x=grouped["diff_bin"].astype(str),
                y=grouped["iv_mean"],
                # mode="markers+text",
                error_y=dict(
                    type="data", array=grouped["iv_std"].tolist(), visible=True
                ),
                text=grouped["count"].astype(int),
                # textposition="top center",
                name=severity,
            ),
            row=row,
            col=1,
        )
        fig.update_yaxes(title="IV days (mean ± SD)", row=row, col=1)
    fig.update_xaxes(title=f"{diff_col} binned", row=1, col=1)

    title = f"bFEV1_2016_19_IVdays_2019-21_binned_{prctile}th_prctile"

    fig.update_layout(
        xaxis_title=diff_col,
        title=title,
        height=800,
    )
    fig.write_image(dh.get_path_to_main() + f"PlotsCFR/Viz diff vs IV days/{title}.pdf")

50th percentile of absolute difference: 0.457945218826417
60th percentile of absolute difference: 2.781535403334735
70th percentile of absolute difference: 5.978852663315695
80th percentile of absolute difference: 12.69587666933772
90th percentile of absolute difference: 25.269813601145305
